# Build a Re-Ranking Pipeline

A runnable companion to the course project [*Build a Re-Ranking Pipeline*](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/reranking-pipeline). Retrieve fast with a tiny keyword scorer, then re-rank the top-K hits with a stronger cross-encoder model, and benchmark whether it actually helps.

**No API key, no LLM, no `.env`.** The only download is the cross-encoder model (~80MB) on its first run. Works the same in Google Colab, Kaggle Notebooks, or Binder.


## Setup: install dependencies

These are the packages the local example project (`examples/reranking-pipeline/pyproject.toml`) declares, plus `pandas` for the benchmark table.

In [ ]:
!pip install -q sentence-transformers numpy pandas


## Step 1: The corpus and test queries

25 short passages on varied topics, deliberately chosen so the same word means different things in different passages: Python the language vs. the snake, Mercury the planet vs. the god vs. the metal, a river bank vs. a money bank. Those are exactly the cases where a fast keyword stage picks a plausible-but-wrong top hit -- and the cross-encoder can fix the ranking. This is the same `data/corpus/*.txt` folder and `data/test_queries.json` from the local example, embedded inline so there's nothing to upload. Each test query is labeled with the passage id that truly answers it -- the ground truth we'll measure precision against.

In [ ]:
CORPUS = {
    'python_programming.txt': 'Python is a high-level programming language known for its readable syntax and a huge ecosystem of libraries.',
    'python_snake.txt': 'The python is a large non-venomous snake that kills its prey by constriction, squeezing it until it can no longer breathe.',
    'apple_orchard.txt': 'Apple trees grow in orchards, and their fruit ripens in late summer before it is picked.',
    'apple_iphone.txt': 'Apple designs the iPhone, an iconic smartphone released every September since 2007.',
    'river_bank.txt': 'A river bank is the strip of land along the water, where plants and trees often take root.',
    'money_bank.txt': 'A bank is a business that keeps your money safe, pays interest on savings, and issues loans.',
    'bat_animal.txt': 'Bats are the only mammals that can truly fly, using echolocation to hunt insects at night.',
    'baseball_bat.txt': 'A baseball bat is a wooden club a hitter swings to send the ball into the outfield.',
    'mercury_planet.txt': 'Mercury is the smallest planet in the solar system and the closest one to the Sun.',
    'mercury_god.txt': 'In Roman mythology, Mercury was the messenger god, famous for his winged sandals and helmet.',
    'mercury_element.txt': 'Mercury is the only metal that is liquid at room temperature, once used in thermometers.',
    'jupiter.txt': 'Jupiter is the largest planet in our solar system, a gas giant with a storm called the Great Red Spot.',
    'microchip.txt': 'A microchip is a tiny integrated circuit that powers modern computers and smartphones.',
    'potato_chips.txt': 'Potato chips are thin, crispy slices of fried potato, usually salted for a snack.',
    'great_barrier_reef.txt': "The Great Barrier Reef is the world's largest coral reef system, off the coast of Australia.",
    'photosynthesis.txt': 'Photosynthesis lets plants use sunlight to make food and release oxygen from their leaves.',
    'great_wall_china.txt': 'The Great Wall of China was built over centuries to protect against invasions from the north.',
    'mount_everest.txt': 'Mount Everest is the highest mountain on Earth, at about 8,849 meters above sea level.',
    'black_holes.txt': 'Black holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape.',
    'amazon_rainforest.txt': "The Amazon rainforest produces about a fifth of the oxygen in Earth's atmosphere.",
    'beethoven.txt': 'Beethoven, the German composer, wrote his famous Ninth Symphony while completely deaf.',
    'nile_river.txt': 'The Nile is traditionally considered the longest river in the world.',
    'honeybee.txt': 'A honeybee colony has a single queen, thousands of workers, and a few hundred drones.',
    'earth_year.txt': 'The Earth orbits the Sun once every 365 days, which is what defines a year.',
    'dna.txt': 'DNA is the molecule that carries the genetic instructions for all living things.',
}

TEST_QUERIES = [
    {'query': 'What is the biggest planet in the solar system?', 'relevant': ['jupiter.txt']},
    {'query': 'Which planet in our solar system is closest to the Sun?', 'relevant': ['mercury_planet.txt']},
    {'query': 'How does a computer run Python code?', 'relevant': ['python_programming.txt']},
    {'query': 'Which snake squeezes its prey to death?', 'relevant': ['python_snake.txt']},
    {'query': 'What animal flies at night and hunts insects?', 'relevant': ['bat_animal.txt']},
    {'query': 'Which company makes the iPhone smartphone?', 'relevant': ['apple_iphone.txt']},
    {'query': 'What metal is liquid at room temperature?', 'relevant': ['mercury_element.txt']},
    {'query': 'Where can you keep your money and earn interest?', 'relevant': ['money_bank.txt']},
    {'query': 'What is the tallest mountain in the world?', 'relevant': ['mount_everest.txt']},
    {'query': 'Which river is the longest in the world?', 'relevant': ['nile_river.txt']},
    {'query': 'What molecule carries genetic information?', 'relevant': ['dna.txt']},
    {'query': 'What do plants use sunlight to make?', 'relevant': ['photosynthesis.txt']},
]

documents = [{"id": doc_id, "text": text} for doc_id, text in CORPUS.items()]
print(f"{len(documents)} documents in the corpus, {len(TEST_QUERIES)} test queries")


## Step 2: The fast stage -- keyword overlap

Stage 1 is intentionally naive and cheap: tokenize the query, drop stopwords, and count how much each document's words overlap with it. Pure Python plus one NumPy call -- milliseconds per query, no model. `np.argpartition` finds the top-k without sorting the whole array, so the stage stays fast no matter how big the corpus gets.

In [ ]:
STOPWORDS = frozenset(
    """
    a an and are as at be by for from how in is it of on or that the this
    to was what when where which who with you your
    """.split()
)

import re
from collections import Counter

def tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return Counter(tok for tok in tokens if tok not in STOPWORDS)

def tokens_overlap(query_tokens, doc_tokens):
    """Naive 3-character prefix match: 'hunts' and 'hunt' count as the same
    word. Intentionally crude -- this is the *cheap* stage."""
    overlap = 0.0
    for q_tok, q_count in query_tokens.items():
        for d_tok, d_count in doc_tokens.items():
            if q_tok == d_tok or q_tok[:3] == d_tok[:3]:
                overlap += q_count * min(q_count, d_count)
                break
    return overlap

import numpy as np

def lexical_scores(query, documents):
    qt = tokenize(query)
    return np.array([tokens_overlap(qt, tokenize(doc["text"])) for doc in documents])

def top_k_indices(scores, k):
    if k >= len(scores):
        return np.argsort(scores)[::-1]
    idx = np.argpartition(scores, -k)[-k:]
    return idx[np.argsort(scores[idx])[::-1]]

def retrieve_fast(query, documents, top_k=5):
    """Fast stage only: keyword overlap, no re-ranking."""
    scores = lexical_scores(query, documents)
    return [(float(scores[i]), documents[i]) for i in top_k_indices(scores, top_k)]


## Step 3: Run the fast stage, see it go wrong

The fast stage retrieves the *right document* into its shortlist -- but not always at position 1. It cannot tell that "biggest" and "largest" are related (Mercury is the *smallest* planet, Jupiter the *largest*; both are "the ...est planet in the solar system"), nor that a "computer running Python" is about the programming language and not the snake.

In [ ]:
def show(query):
    ranking = retrieve_fast(query, documents)
    rel = next(t["relevant"] for t in TEST_QUERIES if t["query"] == query)
    print("Query:", query)
    for i, (score, doc) in enumerate(ranking[:3], 1):
        mark = "  <-- relevant" if doc["id"] in rel else ""
        print(f"  {i}. {score:6.2f}  {doc['id']}{mark}")

show("What is the biggest planet in the solar system?")
show("How does a computer run Python code?")
show("Which company makes the iPhone smartphone?")


## Step 4: The re-ranker -- a cross-encoder

Stage 2 is slow but far more accurate. A **cross-encoder** reads the query and each candidate document *together* as a pair and scores how well the document answers the query -- the model itself was trained on exactly this task (MS MARCO). Because it compares the two texts directly instead of comparing two separately-computed vectors, it captures meaning the keyword stage can't. The catch: it must run once per (query, document) pair, so you only run it on the top-K shortlist. The cell below loads `cross-encoder/ms-marco-MiniLM-L-6-v2` (the same model as the local example; ~80MB on first run) and re-ranks the three queries from Step 3 side by side.

In [ ]:
from sentence_transformers import CrossEncoder
import time

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
model = CrossEncoder(MODEL_NAME)  # downloads ~80MB on first run

def rerank(query, candidates):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = model.predict(pairs)
    ranked = sorted(zip(map(float, scores), candidates), key=lambda t: t[0], reverse=True)
    return ranked

def retrieve_full(query, documents, top_k=5):
    """Fast stage + cross-encoder re-ranking of the shortlist."""
    scores = lexical_scores(query, documents)
    shortlist = [documents[i] for i in top_k_indices(scores, top_k)]
    return rerank(query, shortlist)

for q in ["What is the biggest planet in the solar system?",
          "How does a computer run Python code?",
          "Which company makes the iPhone smartphone?"]:
    fast = retrieve_fast(q, documents)
    smart = retrieve_full(q, documents)
    rel = next(t["relevant"] for t in TEST_QUERIES if t["query"] == q)
    print("\nQuery:", q)
    print("  fast-only top-3 :", [d["id"] for _, d in fast[:3]])
    print("  re-ranked top-3 :", [d["id"] for _, d in smart[:3]])
    print("  relevant        :", rel)


## Step 5: The benchmark -- does re-ranking pay for itself?

Run all 12 queries through both pipelines and measure precision@1, precision@3, and time per query with pandas. The honest picture this project exists to show: re-ranking is a **compute/quality tradeoff**. On this small, well-matched corpus the fast stage already finds the answer in its top-3 for every query, so precision@3 is already 1.0 -- the win is *ordering* (precision@1). And the re-ranker costs on the order of a hundred times more per query.

In [ ]:
import pandas as pd

rows = []
for item in TEST_QUERIES:
    q = item["query"]
    relevant = set(item["relevant"])

    t0 = time.perf_counter()
    fast = retrieve_fast(q, documents)
    fast_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    smart = retrieve_full(q, documents)
    smart_ms = (time.perf_counter() - t0) * 1000

    rows.append({
        "query": q,
        "fast top1": fast[0][1]["id"],
        "rerank top1": smart[0][1]["id"],
        "fast p@1": int(fast[0][1]["id"] in relevant),
        "rerank p@1": int(smart[0][1]["id"] in relevant),
        "fast p@3": int(any(d["id"] in relevant for _, d in fast[:3])),
        "rerank p@3": int(any(d["id"] in relevant for _, d in smart[:3])),
        "fast ms": round(fast_ms, 1),
        "rerank ms": round(smart_ms, 1),
    })

df = pd.DataFrame(rows)
summary = pd.DataFrame({
    "p@1": [df["fast p@1"].mean(), df["rerank p@1"].mean()],
    "p@3": [df["fast p@3"].mean(), df["rerank p@3"].mean()],
    "avg ms/query": [df["fast ms"].mean(), df["rerank ms"].mean()],
}, index=["fast-only (keyword)", "fast + re-rank"])
print(df.to_string(index=False))
print("\nBenchmark summary")
print(summary.round(2).to_string())
print(f"\nRe-ranking improved p@1 by +{df['rerank p@1'].mean() - df['fast p@1'].mean():.2f} "
      f"at {df['rerank ms'].mean() / max(df['fast ms'].mean(), 1e-9):.0f}x the per-query cost.")


## What you just built

A two-stage retrieval pipeline: a milliseconds-fast lexical stage that gets you into the right neighborhood, and a cross-encoder that spends real compute re-reading the top few candidates to get the ordering right -- the same architecture production search systems use, just with 25 passages instead of millions.

Two honest limitations worth remembering:

- **Re-ranking only fixes what it's given.** If the fast stage's shortlist never contains the right document, no re-ranker can rescue it -- which is why production systems tune their first stage too.
- **On a bigger, noisier corpus the gap grows.** More documents means more plausible-but-wrong candidates, which is where re-ranking's precision gains really show up.

Next: swap the keyword stage for the embedding-based search from the *RAG App Over Your Own Notes* project (`all-MiniLM-L6-v2` + `numpy`), or try a stronger cross-encoder like `cross-encoder/ms-marco-electra-base`. The pipeline shape stays the same either way.